# 02 — Getting Started with PySpark

**Environment:** Ubuntu 22.04 · Python 3.12.3 · OpenJDK 17 · Spark 4.1.x · local PySpark.

Run cells in order. Each notebook can start a Spark session independently.

**Learning goal:** Start Spark, inspect its version, load CSV/JSON/TSV and Parquet, and inspect schemas. TSV/CSV fields are initially read as strings to make data-quality checks explicit.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'transactions.csv').is_file()), None)
assert ROOT is not None, 'Open the course folder in Jupyter before running this notebook.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from training_lib import create_spark, read_raw, typed_transactions, quality_condition, clean_transactions, completed_enriched, revenue_by_category
from pyspark.sql import functions as F
spark = create_spark('PySpark-Course')
print('Spark:', spark.version, '| Python:', sys.version.split()[0], '| master:', spark.sparkContext.master)

In [ ]:
transactions, customers, products = read_raw(spark, ROOT)
transactions.printSchema()
transactions.show(5, truncate=False)
print('Raw transactions:', transactions.count())

### JSON records and TSV

The supplied JSON is newline-delimited (one JSON object per line). The TSV file uses tab separators.

In [ ]:
products.printSchema()
products.show(5, truncate=False)
products_tsv = (spark.read.option('header', True).option('sep', '\t')
                .csv(str(ROOT / 'data' / 'products.tsv')))
products_tsv.show(5, truncate=False)

### A DataFrame created from Python values

In [ ]:
small = spark.createDataFrame([('Pune', 4), ('Mumbai', 7)], ['city', 'orders'])
small.printSchema()
small.show()

### Write a Parquet dataset and read it back

Spark writes **directories of part files**, not a single Parquet file. `mode("overwrite")` is used only against the dedicated output folder.

In [ ]:
out = ROOT / 'output' / 'products_parquet'
products.write.mode('overwrite').parquet(str(out))
restored = spark.read.parquet(str(out))
restored.printSchema()
restored.show(5, truncate=False)
assert restored.count() == products.count() == 12

**Check:** What is the difference between `printSchema()`, `show()`, and `count()`? Why do we avoid `inferSchema=True` when malformed CSV values must remain observable?